# 인덕션 헤드: 인컨텍스트 학습의 비밀 - 인덕션 헤드 메커니즘

- Tutorial ID: `tut-4-1`
- Tutorial: 인덕션 헤드: 인컨텍스트 학습의 비밀
- Section ID: `s4-1-1`
- Section: 인덕션 헤드 메커니즘


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np

print("=" * 60)
print("인덕션 헤드 (Induction Head) 시뮬레이션")
print("=" * 60)

np.random.seed(0)

def softmax(x, axis=-1):
        x = x - np.max(x, axis=axis, keepdims=True)
    return np.exp(x) / np.sum(np.exp(x), axis=axis, keepdims=True)

In [ ]:
# 시퀀스 설정

In [ ]:
# 패턴: A B C D A B C ? → 예측: D
vocab = ['<PAD>', 'A', 'B', 'C', 'D', 'E']
V = len(vocab)
d_model = 8
d_head = 4

# 반복 패턴이 있는 시퀀스
sequence = [1, 2, 3, 4, 1, 2, 3]  # A B C D A B C
seq_len = len(sequence)

print(f"시퀀스: {[vocab[t] for t in sequence]}")
print(f"목표: 다음 토큰 예측 → '{vocab[4]}' (D)")
print()

# 임베딩 (단순화: 단위 행렬 기반)
W_E = np.eye(V, d_model) + np.random.randn(V, d_model) * 0.05
W_U = np.random.randn(d_model, V) * 0.3

X = W_E[sequence]  # (seq_len, d_model)

In [ ]:
# 레이어 0: 이전 토큰 헤드 (Previous Token Head)

In [ ]:
print("=" * 60)
print("레이어 0: 이전 토큰 헤드 시뮬레이션")
print("=" * 60)

# 이전 토큰 헤드: 위치 i는 주로 위치 i-1을 주목
# 이것을 구현하는 W_Q, W_K 설계
W_Q0 = np.random.randn(d_model, d_head) * 0.3
W_K0 = np.random.randn(d_model, d_head) * 0.3

# 위치 편향 추가 (이전 위치를 선호하도록)
# 실제 모델에서는 훈련으로 학습됨
pos_embeddings = np.zeros((seq_len, d_model))
for i in range(seq_len):
    pos_embeddings[i, i % d_model] = 1.0  # 단순화된 위치 임베딩

X_with_pos = X + pos_embeddings * 0.5

Q0 = X_with_pos @ W_Q0
K0 = X_with_pos @ W_K0

# 이전 토큰 헤드: 이전 위치에 높은 점수 부여
attn_scores_0 = Q0 @ K0.T / np.sqrt(d_head)

# 오프셋 1 위치에 높은 가중치 (이전 토큰 선호 시뮬레이션)
for i in range(seq_len):
    if i > 0:
        attn_scores_0[i, i-1] += 3.0  # 이전 토큰 강조

# 인과적 마스크
mask = np.triu(np.full((seq_len, seq_len), -1e9), k=1)
A0 = softmax(attn_scores_0 + mask)

print("레이어 0 어텐션 패턴 (이전 토큰 헤드):")
print("     " + " ".join(f"{vocab[t]:4s}" for t in sequence))
for i, t in enumerate(sequence):
        weights = " ".join(f"{A0[i,j]:.2f}" for j in range(seq_len))
    print(f"  {vocab[t]}: {weights}")
print("↑ 각 행: 해당 위치가 이전 위치를 주목함을 확인!")

# OV 회로 (이전 토큰을 복사)
W_V0 = np.random.randn(d_model, d_head) * 0.3
W_O0 = np.random.randn(d_head, d_model) * 0.3

V0 = X @ W_V0
head0_out = (A0 @ V0) @ W_O0  # (seq_len, d_model)

In [ ]:
# 레이어 1: 인덕션 헤드 (K-합성 사용)

In [ ]:
print("
" + "=" * 60)
print("레이어 1: 인덕션 헤드 (K-합성)")
print("=" * 60)

# 잔차 스트림: 원래 임베딩 + 레이어0 출력
X1 = X + head0_out  # (seq_len, d_model)

W_Q1 = np.random.randn(d_model, d_head) * 0.3
W_K1 = np.random.randn(d_model, d_head) * 0.3

Q1 = X1 @ W_Q1  # 쿼리: 현재 토큰 + 레이어0 컨텍스트
K1 = X1 @ W_K1  # 키: 이전 토큰 정보 포함!

attn_scores_1 = Q1 @ K1.T / np.sqrt(d_head)

# 인덕션 헤드 시뮬레이션:
# "현재 위치의 토큰 = A인데, 이전에 A 다음에 B가 왔던 위치를 찾아라"
# K-합성: 레이어0이 '이전 토큰' 정보를 키에 주입
# 현재 위치 i에서 "이전에 sequence[i]가 나온 직후 위치"를 찾음
for i in range(seq_len):
        for j in range(i):
        # 위치 j가 sequence[i]와 같은 토큰이면 높은 점수
        if j > 0 and sequence[j-1] == sequence[i]:
            attn_scores_1[i, j] += 4.0  # 인덕션 헤드 시뮬레이션

A1 = softmax(attn_scores_1 + mask)

print("레이어 1 어텐션 패턴 (인덕션 헤드):")
print("     " + " ".join(f"{vocab[t]:4s}" for t in sequence))
for i, t in enumerate(sequence):
        weights = " ".join(f"{A1[i,j]:.2f}" for j in range(seq_len))
    print(f"  {vocab[t]}: {weights}")

print("
마지막 'C' (위치 6)의 어텐션:")
last_attn = A1[-1]
for j, (tok_id, w) in enumerate(zip(sequence, last_attn)):
    if w > 0.05:
        print(f"  위치 {j} ({vocab[tok_id]}): {w:.3f} {'← 이곳 다음이 D!' if tok_id == 3 and j < seq_len-1 else ''}")

print("
인덕션 헤드는 이전 'C'(위치 2) 다음에 'D'가 왔음을 기억!")
print("→ 현재 'C' 다음에도 'D'를 예측합니다.")

In [ ]:
# 경로 확장 (Path Expansion)

In [ ]:
print("
" + "=" * 60)
print("경로 확장: 인덕션 헤드의 역학")
print("=" * 60)
print("""
토큰 A가 위치 t에 등장하고, 이전에 패턴 [..., A, B, ...]가 있었다면:

레이어 0 (이전 토큰 헤드):
  x_1[j+1] += A0_{j+1,j} · W_OV^0 · x_0[j]
  → 위치 j+1에 "이전 토큰 j의 정보"를 기록

레이어 1 (인덕션 헤드, K-합성):
  K1[j+1] = W_K1 · (x_0[j+1] + head0_out[j+1])
           = W_K1 · x_0[j+1] + W_K1 · W_OV^0 · x_0[j]
           ↑ 직접 임베딩   ↑ K-합성된 이전 토큰 정보

  Q1[t] = W_Q1 · x_0[t]  (현재 위치 A의 쿼리)
  
  만약 x_0[t] ≈ x_0[j] (현재 A ≈ 이전 A),
  그리고 W_K1 · W_OV^0 · x_0[j] ≈ W_Q1 · x_0[t]:
  → 위치 j+1 (=B의 위치)에 높은 어텐션!
  → B의 정보를 현재 위치 t에 복사 → 다음 토큰 B 예측!
""")